In [ ]:
import pandas as pd

# Raw file stays unchanged; cleaning works on a loaded copy
df = pd.read_csv("../data/raw/netflix_titles.csv")

print(df.shape)
df.info()
print(df.isna().sum())

## Step 2: Cleaning rules

**Test used:** could this field have a value for this title? If yes, a blank means *unknown*. If no, it means *not applicable*. Missing values are kept as true missing values (not the text "NA") and are never filled with invented data.

| Column | Blanks | Meaning | Decision | Reason |
|---|---|---|---|---|
| director | 2,634 | Unknown | Keep as missing | Blank for 2,446 TV shows and 188 movies. Both can have directors, so the blanks are missing source data. Filling would invent names. |
| cast | 825 | Unknown | Keep as missing | Every title can have a cast. Filling would invent names. |
| country | 831 | Unknown | Keep as missing; excluded from country analysis | Every title can have a country. Filling would invent a location. |
| date_added | 10 | Unknown | Keep row; date stays missing | Only 10 of 8,807 rows. Other fields in these rows are still useful. |
| rating | 7 (was 4) | Unknown | Keep as missing | Three of these are the rows where the runtime was moved out. The true rating isn't in the data, so filling would invent one. |
| duration | 0 (was 3) | Unknown | Fixed | Three titles (Louis C.K. specials) had their runtime in the rating column. Moved to duration. |


In [ ]:
# Remove stray spaces, then convert the text to a date
df["date_added"] = pd.to_datetime(df["date_added"].str.strip())

# Pull out the year as its own column for trend analysis
df["year_added"] = df["date_added"].dt.year

In [ ]:
print(df["date_added"].dtype)
print(df["date_added"].isna().sum())
print(df[["date_added", "year_added"]].head())

In [ ]:
# Pull the number in front of "min" into a movie-only column
df["duration_mins"] = pd.to_numeric(df["duration"].str.extract(r"(\d+)\s*min")[0])

# Pull the number in front of "Season" into a show-only column
df["duration_seasons"] = pd.to_numeric(df["duration"].str.extract(r"(\d+)\s*Season")[0])

In [ ]:
# Count non-blank values in each new column, split by type
df.groupby("type")[["duration_mins", "duration_seasons"]].count()

In [ ]:
# Show the rows where duration is blank, with the fields that might hold the value by mistake
df[df["duration"].isna()][["show_id", "type", "title", "rating", "duration"]]

In [ ]:
# Rows where rating holds a runtime such as "74 min" (the duration was entered in the wrong column)
misplaced = df["rating"].str.contains(r"\d+\s*min", na=False)

# Move the runtime into duration, then clear rating because its real value isn't in the data
df.loc[misplaced, "duration"] = df.loc[misplaced, "rating"]
df.loc[misplaced, "rating"] = pd.NA

In [ ]:
# Confirm the three rows now have a duration and a blank rating
print(df.loc[misplaced, ["title", "rating", "duration"]])
print(df["rating"].isna().sum())

In [ ]:
# Check the three moved titles by name, which doesn't depend on the rating column
df[df["title"].str.contains("Louis C.K.", na=False)][["title", "rating", "duration"]]

In [ ]:
# Rerun the split so the three moved movies are included
df["duration_mins"] = pd.to_numeric(df["duration"].str.extract(r"(\d+)\s*min")[0])
df["duration_seasons"] = pd.to_numeric(df["duration"].str.extract(r"(\d+)\s*Season")[0])

# Duration should now have no blanks
print(df["duration"].isna().sum())

In [ ]:
# Remove leading and trailing spaces from every text column
text_cols = df.select_dtypes(include="object").columns
df[text_cols] = df[text_cols].apply(lambda col: col.str.strip())

In [ ]:
print(df.isna().sum())

In [ ]:
# Check for repeated show IDs, which should never happen
print(df["show_id"].duplicated().sum())

# Check for repeated title and type pairs
print(df.duplicated(subset=["title", "type"]).sum())

In [ ]:
# Show every row that shares a title and type with another row
df[df.duplicated(subset=["title", "type"], keep=False)][
    ["show_id", "title", "type", "release_year", "country", "duration"]
]

In [ ]:
# Compare the remaining text fields for the two matching rows
df[df["title"] == "Consequences"][["show_id", "director", "cast", "listed_in"]]

In [ ]:
rows_before = len(df)

# Keep the first copy of each title that matches on all the fields we compared
df = df.drop_duplicates(
    subset=["title", "type", "release_year", "country", "duration"],
    keep="first"
).reset_index(drop=True)

print(f"Removed {rows_before - len(df)} duplicate row(s). Rows now: {len(df)}")

In [ ]:
# Confirm the two Consequences rows match on the remaining fields before we drop one
print(df[df["title"] == "Consequences"][["show_id", "director", "cast", "listed_in"]])

In [ ]:
# Read the untouched raw file and compare the two copies side by side
raw = pd.read_csv("../data/raw/netflix_titles.csv")
raw[raw["show_id"].isin(["s3372", "s6530"])][
    ["show_id", "title", "director", "cast", "country", "date_added", "release_year", "listed_in", "description"]
]

In [ ]:
# Show full text, not truncated, then compare the two descriptions exactly
pd.set_option("display.max_colwidth", None)

raw = pd.read_csv("../data/raw/netflix_titles.csv")
pair = raw[raw["show_id"].isin(["s3372", "s6530"])]

print(pair[["show_id", "description"]].to_string())
print("Descriptions identical:", pair["description"].iloc[0] == pair["description"].iloc[1])

**Duplicates:** "Consequences" (2014, Turkey, Ozan Açıktan) appears twice with identical values in every field, including the description. One copy (s6530) was removed.

In [ ]:
# Save the cleaned table to the processed folder, leaving the raw file untouched
df.to_csv("../data/processed/netflix_clean.csv", index=False)

print(df.shape)